# CREATE SCHEMA 'SILVER'

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.silver;

# CREATE SCHEMA 'QUARANTINE'

In [0]:
CREATE SCHEMA IF NOT EXISTS parcel_sorting_hub.quarantine;

# CREATING SILVER TABLES

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.devices (
    device_id   STRING,
    type        STRING,
    zone        STRING,
    status      STRING,
    _updated_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.exceptions (
    exception_id STRING,
    parcel_id    DECIMAL(24,0),
    device_id    STRING,
    error_code   STRING,
    status       STRING,
    reported_at  TIMESTAMP,
    resolved_at  TIMESTAMP,
    employee_id  STRING,
    _updated_at  TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.intake (
    delivery_id   STRING,
    dock_id       STRING,
    arrival_time  TIMESTAMP,
    unload_start  TIMESTAMP,
    unload_end    TIMESTAMP,
    parcel_count  INT,
    truck_plate   STRING,
    source_hub_id STRING,
    _updated_at   TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.loading (
    loading_id    STRING,
    dock_id       STRING,
    start_time    TIMESTAMP,
    close_time    TIMESTAMP,
    parcel_count  INT,
    route_id      STRING,
    employee_id   STRING,
    _updated_at   TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.parcels (
    parcel_id           DECIMAL(24,0),
    status              STRING,
    weight_kg           DECIMAL(10,2),
    length_cm           DECIMAL(10,2),
    width_cm            DECIMAL(10,2),
    height_cm           DECIMAL(10,2),
    service_type        STRING,
    received_at         TIMESTAMP,
    source_hub_id       STRING,
    destination_hub_id  STRING,
    loading_id          STRING,
    _updated_at         TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.scans (
    scan_id        STRING,
    parcel_id      DECIMAL(24,0),
    device_id      STRING,
    scan_type      STRING,
    result         STRING,
    scan_timestamp TIMESTAMP,
    dynamic_weight DECIMAL(10,2),
    _updated_at    TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.sorting (
    event_id    STRING,
    parcel_id   DECIMAL(24,0),
    sorter_id   STRING,
    chute_id    STRING,
    entry_time  TIMESTAMP,
    result      STRING,
    _updated_at TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.silver.status_history (
    history_id       STRING,
    parcel_id        DECIMAL(24,0),
    status           STRING,
    change_timestamp TIMESTAMP,
    _updated_at      TIMESTAMP
);

# CREATING QUARANTINE TABLES

In [0]:
CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.devices (
    device_id          STRING,
    type               STRING,
    zone               STRING,
    status             STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.exceptions (
    exception_id       STRING,
    parcel_id          STRING,
    device_id          STRING,
    error_code         STRING,
    status             STRING,
    reported_at        STRING,
    resolved_at        STRING,
    employee_id        STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.intake (
    delivery_id        STRING,
    dock_id            STRING,
    arrival_time       STRING,
    unload_start       STRING,
    unload_end         STRING,
    parcel_count       STRING,
    truck_plate        STRING,
    source_hub_id      STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.loading (
    loading_id         STRING,
    dock_id            STRING,
    start_time         STRING,
    close_time         STRING,
    parcel_count       STRING,
    route_id           STRING,
    employee_id        STRING,
    _rejection_reason  STRING,
    _rejected_at       TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.parcels (
    parcel_id           STRING,
    status              STRING,
    weight_kg           STRING,
    length_cm           STRING,
    width_cm            STRING,
    height_cm           STRING,
    service_type        STRING,
    received_at         STRING,
    source_hub_id       STRING,
    delivery_id         STRING,
    destination_hub_id  STRING,
    loading_id          STRING,
    _rejection_reason   STRING,
    _rejected_at        TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.scans (
    scan_id           STRING,
    parcel_id         STRING,
    device_id         STRING,
    scan_type         STRING,
    result            STRING,
    scan_timestamp    STRING,
    dynamic_weight    STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.sorting (
    event_id          STRING,
    parcel_id         STRING,
    sorter_id         STRING,
    chute_id          STRING,
    entry_time        STRING,
    result            STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
);


CREATE TABLE IF NOT EXISTS parcel_sorting_hub.quarantine.status_history (
    history_id        STRING,
    parcel_id         STRING,
    status            STRING,
    change_timestamp  STRING,
    _rejection_reason STRING,
    _rejected_at      TIMESTAMP
);

# ORDER OF EXECUTION (table dependencies)

STEP 1 (tables without FK's)
-     1. quarantine.device
-     2. quarantine.intake
-     3. quarantine.loading

STEP 2 (tables with FK's - depends on step 1)
-     4. quarantine.parcels        (FK -> intakes, loading)

STEP 3 (tables with FK's - depends on step 2)
-     5. quarantine.scans          (FK -> parcels, devices)
-     6. quarantine.exceptions     (FK -> parcels, devices)
-     7. quarantine.sorting        (FK -> parcels, devices)
-     8. quarantine.status_history (FK -> parcels)

# DATA VALIDATION (QUARANTINE DATA ISOLATION)

## STEP 1

### DEVICES

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.devices
SELECT 
    device_id,        
    type,            
    zone,             
    status,                    
    CASE
        WHEN device_id IS NULL OR TRIM(device_id) = '' THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(device_id)) = 'null'           THEN 'STRING_NULL_IN_PK'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.devices  
WHERE _ingested_at > (SELECT MAX(_updated_at) FROM parcel_sorting_hub.silver.devices)
AND (
        device_id IS NULL
    OR  TRIM(device_id) = ''
    OR  LOWER(TRIM(device_id)) = 'null'
  )
AND status IN ('ACTIVE', 'MAINTENANCE')
AND type IN ('Sorter', 'Telescopic Conveyor', 'Scanner', 'Dimensioner')
AND zone IN ('Sort_Area', 'Inbound', 'Outbound')
;

### INTAKE

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.intake
SELECT 
    delivery_id,        
    dock_id,            
    arrival_time,             
    unload_start,                    
    unload_end,
    parcel_count,
    truck_plate,
    source_hub_id,
    CASE
        WHEN delivery_id IS NULL OR TRIM(delivery_id) = ''  THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(delivery_id)) = 'null'              THEN 'STRING_NULL_IN_PK'

        WHEN dock_id IS NULL OR TRIM(dock_id) = ''          THEN 'NULL_OR_EMPTY_DOCK_ID'
        WHEN LOWER(TRIM(dock_id)) = 'null'                  THEN 'STRING_NULL_IN_DOCK_ID'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.intake
WHERE _ingested_at > (SELECT MAX(_updated_at) FROM parcel_sorting_hub.silver.intake)
AND (
        delivery_id IS NULL
    OR  TRIM(delivery_id) = ''
    OR  LOWER(TRIM(delivery_id)) = 'null'

    OR  dock_id IS NULL
    OR  TRIM(dock_id) = ''
    OR  LOWER(TRIM(dock_id)) = 'null'
  )
AND arrival_time < unload_start
AND arrival_time < unload_end
AND unload_start < unload_end
AND parcel_count > 0;


### LOADING

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.loading
SELECT 
    loading_id,        
    dock_id,            
    start_time,             
    close_time,                    
    parcel_count,
    route_id,
    employee_id,
    CASE
        WHEN loading_id IS NULL OR TRIM(loading_id) = ''    THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(loading_id)) = 'null'               THEN 'STRING_NULL_IN_PK'

        WHEN dock_id IS NULL OR TRIM(dock_id) = ''          THEN 'NULL_OR_EMPTY_DOCK_ID'
        WHEN LOWER(TRIM(dock_id)) = 'null'                  THEN 'STRING_NULL_IN_DOCK_ID'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.loading l
WHERE _ingested_at > (SELECT MAX(_updated_at) FROM parcel_sorting_hub.silver.loading)
AND (
        loading_id IS NULL
    OR  TRIM(loading_id) = ''
    OR  LOWER(TRIM(loading_id)) = 'null'

    OR  dock_id IS NULL
    OR  TRIM(dock_id) = ''
    OR  LOWER(TRIM(dock_id)) = 'null'
  )
AND start_time < close_time
AND parcel_count > 0;


## STEP 2

### PARCELS

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.parcels
SELECT 
    p.parcel_id,        
    p.status,            
    p.weight_kg,             
    p.length_cm,                    
    p.width_cm,
    p.height_cm,
    p.service_type,
    p.received_at,
    p.source_hub_id,
    p.delivery_id,
    p.destination_hub_id,
    p.loading_id,
    CASE
        WHEN p.parcel_id IS NULL OR TRIM(p.parcel_id) = ''          THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(p.parcel_id)) = 'null'                      THEN 'STRING_NULL_IN_PK'

        WHEN p.source_hub_id IS NULL OR TRIM(p.source_hub_id) = ''  THEN 'NULL_OR_EMPTY_SOURCE_HUB_ID'
        WHEN LOWER(TRIM(p.source_hub_id)) = 'null'                  THEN 'STRING_NULL_IN_SOURCE_HUB_ID'

        WHEN p.delivery_id IS NULL OR TRIM(p.delivery_id) = ''      THEN 'NULL_OR_EMPTY_DELIVERY_ID'
        WHEN LOWER(TRIM(p.delivery_id)) = 'null'                    THEN 'STRING_NULL_IN_DELIVERY_ID'

        WHEN p.destination_hub_id IS NULL 
            OR TRIM(p.destination_hub_id) = ''                      THEN 'NULL_OR_EMPTY_DESTINATION_HUB_ID'
        WHEN LOWER(TRIM(p.destination_hub_id)) = 'null'             THEN 'STRING_NULL_IN_DESTINATION_HUB_ID'

        WHEN p.loading_id IS NULL OR TRIM(p.loading_id) = ''        THEN 'NULL_OR_EMPTY_LOADING_ID'
        WHEN LOWER(TRIM(p.loading_id)) = 'null'                     THEN 'STRING_NULL_IN_LOADING_ID'

        WHEN i.delivery_id IS NULL                                  THEN 'FK_DELIVERY_ID_NOT_FOUND_IN_SILVER'
        WHEN l.loading_id IS NULL                                   THEN 'FK_LOADING_ID_NOT_FOUND_IN_SILVER'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.parcels p

LEFT JOIN parcel_sorting_hub.silver.intake i 
    ON p.delivery_id = i.delivery_id

LEFT JOIN parcel_sorting_hub.silver.loading l  
    ON p.loading_id = l.loading_id

WHERE p._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.parcels) 
AND (
        p.parcel_id IS NULL
    OR  TRIM(p.parcel_id) = ''
    OR  LOWER(TRIM(p.parcel_id)) = 'null'

    OR  p.source_hub_id IS NULL
    OR  TRIM(p.source_hub_id) = ''
    OR  LOWER(TRIM(p.source_hub_id)) = 'null'

    OR  p.delivery_id IS NULL
    OR  TRIM(p.delivery_id) = ''
    OR  LOWER(TRIM(p.delivery_id)) = 'null'

    OR  p.destination_hub_id IS NULL
    OR  TRIM(p.destination_hub_id) = ''
    OR  LOWER(TRIM(p.destination_hub_id)) = 'null'

    OR  p.loading_id IS NULL
    OR  TRIM(p.loading_id) = ''
    OR  LOWER(TRIM(p.loading_id)) = 'null'

    OR  i.delivery_id IS NULL
    OR  l.loading_id IS NULL
)
AND p.status IN ('LOADED', 'EXCEPTION')
AND p.weight_kg > 0
AND p.length_cm > 0
AND p.width_cm > 0
AND p.height_cm > 0
AND p.service_type IN ('PRIORITY', 'LOCKER_24H', 'COURIER_STD');

## STEP 3

### SCANS

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.scans
SELECT 
    s.scan_id,        
    s.parcel_id,            
    s.device_id,             
    s.scan_type,   
    s.result,
    s.scan_timestamp,
    s.dynamic_weight,
    CASE
        WHEN s.scan_id IS NULL OR TRIM(s.scan_id) = ''      THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(s.scan_id)) = 'null'                THEN 'STRING_NULL_IN_PK'

        WHEN s.parcel_id IS NULL OR TRIM(s.parcel_id) = ''  THEN 'NULL_OR_EMPTY_PARCEL_ID'
        WHEN LOWER(TRIM(s.parcel_id)) = 'null'              THEN 'STRING_NULL_IN_PARCEL_ID'

        WHEN s.device_id IS NULL OR TRIM(s.device_id) = ''  THEN 'NULL_OR_EMPTY_DEVICE_ID'
        WHEN LOWER(TRIM(s.device_id)) = 'null'              THEN 'STRING_NULL_IN_DEVICE_ID'
        
        WHEN d.device_id IS NULL                            THEN 'FK_DEVICE_NOT_FOUND_IN_SILVER'
        WHEN p.parcel_id IS NULL                            THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.scans s

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON CAST(s.parcel_id AS DECIMAL(24,0)) = p.parcel_id

LEFT JOIN parcel_sorting_hub.silver.devices d  
    ON s.device_id = d.device_id

WHERE s._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.scans)
AND (
        s.scan_id IS NULL
    OR  TRIM(s.scan_id) = ''
    OR  LOWER(TRIM(s.scan_id)) = 'null'

    OR  s.parcel_id IS NULL
    OR  TRIM(s.parcel_id) = ''
    OR  LOWER(TRIM(s.parcel_id)) = 'null'

    OR  s.device_id IS NULL
    OR  TRIM(s.device_id) = ''
    OR  LOWER(TRIM(s.device_id)) = 'null'

    OR  d.device_id IS NULL
    OR  p.parcel_id IS NULL
    )
AND s.scan_type IN ('LOADED', 'INBOUND', 'LOADING_DOCK', 'DIMENSIONING', 'DESTINATION')
AND s.result IN ('OK', 'FAIL')
AND s.dynamic_weight > 0;


### EXCEPTIONS

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.exceptions
SELECT 
    e.exception_id,        
    e.parcel_id,            
    e.device_id,             
    e.error_code,   
    e.status,
    e.reported_at,
    e.resolved_at,
    e.employee_id,               
    CASE

        WHEN e.exception_id IS NULL OR TRIM(e.exception_id) = '' THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(e.exception_id)) = 'null'                THEN 'STRING_NULL_IN_PK'

        WHEN e.device_id IS NULL OR TRIM(e.device_id) = ''       THEN 'NULL_OR_EMPTY_FK'
        WHEN LOWER(TRIM(e.device_id)) = 'null'                   THEN 'STRING_NULL_IN_FK' 
        WHEN d.device_id IS NULL                                 THEN 'FK_DEVICE_NOT_FOUND_IN_SILVER'
        

        WHEN e.parcel_id IS NULL OR TRIM(e.parcel_id) = ''       THEN 'NULL_OR_EMPTY_FK'
        WHEN LOWER(TRIM(e.parcel_id)) = 'null'                   THEN 'STRING_NULL_IN_FK'
        WHEN p.parcel_id IS NULL                                 THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.exceptions e

LEFT JOIN parcel_sorting_hub.silver.devices d 
    ON e.device_id = d.device_id

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON CAST(e.parcel_id AS decimal(24,0)) = p.parcel_id

WHERE e._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.exceptions)
AND (
        e.exception_id IS NULL
    OR  TRIM(e.exception_id) = ''
    OR  LOWER(TRIM(e.exception_id)) = 'null' 

    OR  e.device_id IS NULL
    OR  TRIM(e.device_id) = ''
    OR  LOWER(TRIM(e.device_id)) = 'null'

    OR e.parcel_id IS NULL 
    OR TRIM(e.parcel_id) = '' 
    OR LOWER(TRIM(e.parcel_id)) = 'null'

    OR  d.device_id IS NULL 
    OR  p.parcel_id IS NULL 
  )
AND e.status IN ('RESOLVED', 'OPEN')
AND e.reported_at < e.resolved_at;


### SORTING

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.sorting
SELECT 
    s.event_id,        
    s.parcel_id,            
    s.sorter_id,             
    s.chute_id,   
    s.entry_time,
    s.result,
    CASE
        WHEN s.event_id IS NULL OR TRIM(s.event_id) = ''    THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(s.event_id)) = 'null'               THEN 'STRING_NULL_IN_PK'

        WHEN s.parcel_id IS NULL OR TRIM(s.parcel_id) = ''  THEN 'NULL_OR_EMPTY_PARCEL_ID'
        WHEN LOWER(TRIM(s.parcel_id)) = 'null'              THEN 'STRING_NULL_IN_PARCEL_ID'

        WHEN s.sorter_id IS NULL OR TRIM(s.sorter_id) = ''  THEN 'NULL_OR_EMPTY_SORTER_ID'
        WHEN LOWER(TRIM(s.sorter_id)) = 'null'              THEN 'STRING_NULL_IN_SORTER_ID'

        WHEN s.chute_id IS NULL OR TRIM(s.chute_id) = ''    THEN 'NULL_OR_EMPTY_CHUTE_ID'
        WHEN LOWER(TRIM(s.chute_id)) = 'null'               THEN 'STRING_NULL_IN_CHUTE_ID'

        WHEN p.parcel_id IS NULL                            THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'
        WHEN d.device_id IS NULL                            THEN 'FK_SORTER_NOT_FOUND_IN_SILVER'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.sorting s

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON CAST(s.parcel_id AS DECIMAL(24,0)) = p.parcel_id

LEFT JOIN parcel_sorting_hub.silver.devices d  
    ON s.sorter_id = d.device_id

WHERE s._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.sorting)
AND (
        s.event_id IS NULL
    OR  TRIM(s.event_id) = ''
    OR  LOWER(TRIM(s.event_id)) = 'null'

    OR  s.parcel_id IS NULL
    OR  TRIM(s.parcel_id) = ''
    OR  LOWER(TRIM(s.parcel_id)) = 'null'

    OR  s.sorter_id IS NULL
    OR  TRIM(s.sorter_id) = ''
    OR  LOWER(TRIM(s.sorter_id)) = 'null'

    OR  s.chute_id IS NULL
    OR  TRIM(s.chute_id) = ''
    OR  LOWER(TRIM(s.chute_id))

    OR  p.parcel_id IS NULL
    OR  d.device_id IS NULL
    )
AND s.result IN ('SUCCESS', 'REROUTED');

### STATUS HISTORY

In [0]:
INSERT INTO parcel_sorting_hub.quarantine.status_history
SELECT 
    h.history_id,        
    h.parcel_id,            
    h.status_name,             
    h.change_timestamp,
    CASE
        WHEN h.history_id IS NULL OR TRIM(h.history_id) = ''    THEN 'NULL_OR_EMPTY_PK'
        WHEN LOWER(TRIM(h.history_id)) = 'null'                 THEN 'STRING_NULL_IN_PK'

        WHEN h.parcel_id IS NULL OR TRIM(h.parcel_id) = ''      THEN 'NULL_OR_EMPTY_PARCEL_ID'
        WHEN LOWER(TRIM(h.parcel_id)) = 'null'                  THEN 'STRING_NULL_IN_PARCEL_ID'

        WHEN h.status_name IS NULL OR TRIM(h.status_name) = ''  THEN 'NULL_OR_EMPTY_STATUS_NAME'
        WHEN LOWER(TRIM(h.status_name)) = 'null'                THEN 'STRING_NULL_IN_STATUS_NAME'

        WHEN h.change_timestamp IS NULL 
            OR TRIM(h.change_timestamp) = ''                    THEN 'NULL_OR_EMPTY_CHANGE_TIMESTAMP'
        WHEN LOWER(TRIM(h.change_timestamp)) = 'null'           THEN 'STRING_NULL_IN_CHANGE_TIMESTAMP'

        WHEN p.parcel_id IS NULL                                THEN 'FK_PARCEL_NOT_FOUND_IN_SILVER'
    END AS _rejection_reason,
    current_timestamp() AS _rejected_at
FROM parcel_sorting_hub.bronze.status_history h

LEFT JOIN parcel_sorting_hub.silver.parcels p 
    ON CAST(h.parcel_id AS DECIMAL(24,0)) = p.parcel_id

WHERE h._ingested_at > (SELECT COALESCE(MAX(_updated_at), '2026-01-01') FROM parcel_sorting_hub.silver.status_history)
AND (
        h.history_id IS NULL
    OR  TRIM(h.history_id) = ''
    OR  LOWER(TRIM(h.history_id)) = 'null'

    OR  h.parcel_id IS NULL
    OR  TRIM(h.parcel_id) = ''
    OR  LOWER(TRIM(h.parcel_id)) = 'null'

    OR  h.status_name IS NULL
    OR  TRIM(h.status_name) = ''
    OR  LOWER(TRIM(h.status_name)) = 'null'

    OR  h.change_timestamp IS NULL
    OR  TRIM(h.change_timestamp) = ''
    OR  LOWER(TRIM(h.change_timestamp)) = 'null'

    OR  p.parcel_id IS NULL
    )
AND h.status_name IN ('RECEIVED', 'SORTED', 'LOADED', 'EXCEPTIONS');

# MERGING CLEANED & DEDUPED DATA  

## DEVICES

In [0]:
MERGE INTO parcel_sorting_hub.silver.devices AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.devices
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.devices
        )
            AND device_id IS NOT NULL
            AND TRIM(device_id) != ''
            AND LOWER(TRIM(device_id)) != 'null'
            AND status IN ('ACTIVE', 'MAINTENANCE')
            AND type IN ('Sorter', 'Telescopic Conveyor', 'Scanner', 'Dimensioner')
            AND zone IN ('Sort_Area', 'Inbound', 'Outbound')
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    device_id,
                    type,
                    zone,
                    status
                ORDER BY _ingested_at DESC) row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(device_id)     AS device_id,
        TRIM(type)          AS type,
        TRIM(zone)          AS zone,
        TRIM(status)        AS status,
        current_timestamp() AS _updated_at
    FROM deduped

) AS source
ON target.device_id = source.device_id
WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *

## INTAKE

In [0]:
MERGE INTO parcel_sorting_hub.silver.intake AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.intake
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.intake
        )
            AND delivery_id IS NOT NULL
            AND TRIM(delivery_id) != ''
            AND LOWER(TRIM(delivery_id)) != 'null'
            AND arrival_time < unload_start
            AND arrival_time < unload_end
            AND unload_start < unload_end
            AND parcel_count > 0
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY delivery_id ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    delivery_id,
                    dock_id,
                    arrival_time,
                    unload_start,
                    unload_end,
                    parcel_count,
                    truck_plate,
                    source_hub_id
                ORDER BY _ingested_at DESC) row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(delivery_id)   AS delivery_id,
        TRIM(dock_id)       AS dock_id,
        TRIM(arrival_time)  AS arrival_time,
        TRIM(unload_start)  AS unload_start,
        TRIM(unload_end)    AS unload_end,
        TRIM(parcel_count)  AS parcel_count,
        TRIM(truck_plate)   AS truck_plate,
        TRIM(source_hub_id) AS source_hub_id,
        current_timestamp() AS _updated_at
    FROM deduped

) AS source
ON target.delivery_id = source.delivery_id
WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *

## LOADING

In [0]:
MERGE INTO parcel_sorting_hub.silver.loading AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.loading
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.loading
        )
            AND loading_id IS NOT NULL
            AND TRIM(loading_id) != ''
            AND LOWER(TRIM(loading_id)) != 'null'
            AND start_time < close_time
            AND parcel_count > 0
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY loading_id ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    loading_id,
                    dock_id,
                    start_time,
                    close_time,
                    parcel_count,
                    route_id,
                    employee_id
                ORDER BY _ingested_at DESC) row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(loading_id)    AS loading_id,
        TRIM(dock_id)       AS dock_id,
        TRIM(start_time)    AS start_time,
        TRIM(close_time)    AS close_time,
        TRIM(parcel_count)  AS parcel_count,
        TRIM(route_id)      AS route_id,
        TRIM(employee_id)   AS employee_id,
        current_timestamp() AS _updated_at
    FROM deduped

) AS source
ON target.loading_id = source.loading_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## PARCELS

In [0]:
MERGE INTO parcel_sorting_hub.silver.parcels AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.parcels
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.parcels
        )
            AND parcel_id IS NOT NULL
            AND TRIM(parcel_id) != ''
            AND TRIM(parcel_id) != 'null'
            AND status IN ('LOADED', 'EXCEPTION')
            AND weight_kg > 0
            AND length_cm > 0
            AND width_cm > 0
            AND height_cm > 0
            AND service_type IN ('PRIORITY', 'LOCKER_24H', 'COURIER_STD')
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(parcel_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(parcel_id),
                    TRIM(status),
                    TRIM(weight_kg),
                    TRIM(length_cm),
                    TRIM(width_cm),
                    TRIM(height_cm),
                    TRIM(service_type),
                    TRIM(received_at),
                    TRIM(source_hub_id),
                    TRIM(destination_hub_id),
                    TRIM(loading_id)
                ORDER BY _ingested_at DESC) row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        try_cast(TRIM(parcel_id) AS DECIMAL(24,0))                                  AS parcel_id,
        TRIM(status)                                                                 AS status,
        try_cast(TRIM(weight_kg) AS DECIMAL(10,2))                                  AS weight_kg,
        try_cast(TRIM(length_cm) AS DECIMAL(10,2))                                  AS length_cm,
        try_cast(TRIM(width_cm) AS DECIMAL(10,2))                                   AS width_cm,
        try_cast(TRIM(height_cm) AS DECIMAL(10,2))                                  AS height_cm,
        TRIM(service_type)                                                           AS service_type,
        COALESCE(
            try_to_timestamp(TRIM(received_at), 'dd/MM/yyyy HH:mm'),
            try_to_timestamp(TRIM(received_at), 'yyyy-MM-dd HH:mm:ss')
        )                                                                            AS received_at,
        TRIM(source_hub_id)                                                          AS source_hub_id,
        TRIM(destination_hub_id)                                                     AS destination_hub_id,
        TRIM(loading_id)                                                             AS loading_id,
        current_timestamp()                                                          AS _updated_at
    FROM deduped

) AS source
ON target.parcel_id = source.parcel_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## SCANS

In [0]:
MERGE INTO parcel_sorting_hub.silver.scans AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.scans
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.scans
        )
            AND scan_id IS NOT NULL
            AND TRIM(scan_id) != ''
            AND TRIM(scan_id) != 'null'
            AND scan_type IN ('LOADED', 'INBOUND', 'LOADING_DOCK', 'DIMENSIONING', 'DESTINATION')
            AND result IN ('OK', 'FAIL')
            AND dynamic_weight > 0
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(scan_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(scan_id),
                    TRIM(parcel_id),
                    TRIM(device_id),
                    TRIM(scan_type),
                    TRIM(result),
                    TRIM(scan_timestamp),
                    TRIM(dynamic_weight)
                ORDER BY _ingested_at DESC) row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(scan_id)           AS scan_id,
        TRIM(parcel_id)         AS parcel_id,
        TRIM(device_id)         AS device_id,
        TRIM(scan_type)         AS scan_type,
        TRIM(result)            AS result,
        TRIM(scan_timestamp)    AS scan_timestamp,
        TRIM(dynamic_weight)    AS dynamic_weight,
        current_timestamp()     AS _updated_at
    FROM deduped

) AS source
ON target.scan_id = source.scan_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## EXCEPTIONS

In [0]:
MERGE INTO parcel_sorting_hub.silver.exceptions AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.exceptions
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.exceptions
        )
            AND exception_id IS NOT NULL
            AND TRIM(exception_id) != ''
            AND TRIM(exception_id) != 'null'
            AND status IN ('RESOLVED', 'OPEN')
            AND reported_at < resolved_at
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(exception_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(exception_id),
                    TRIM(parcel_id),
                    TRIM(device_id),
                    TRIM(error_code),
                    TRIM(status),
                    TRIM(reported_at),
                    TRIM(resolved_at),
                    TRIM(employee_id)
                ORDER BY _ingested_at DESC) AS row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(exception_id)  AS exception_id,
        TRIM(parcel_id)     AS parcel_id,
        TRIM(device_id)     AS device_id,
        TRIM(error_code)    AS error_code,
        TRIM(status)        AS status,
        TRIM(reported_at)   AS reported_at,
        TRIM(resolved_at)   AS resolved_at,
        TRIM(employee_id)   AS employee_id,
        current_timestamp() AS _updated_at
    FROM deduped

) AS source
ON target.exception_id = source.exception_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *     

## SORTING

In [0]:
MERGE INTO parcel_sorting_hub.silver.sorting AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.sorting
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.sorting
        )
            AND event_id IS NOT NULL
            AND TRIM(event_id) != ''
            AND TRIM(event_id) != 'null'
            AND result IN ('SUCCESS', 'REROUTED')
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(event_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(event_id),
                    TRIM(parcel_id),
                    TRIM(sorter_id),
                    TRIM(chute_id), 
                    TRIM(entry_time),
                    TRIM(result)
                ORDER BY _ingested_at DESC) AS row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(event_id)      AS event_id,
        TRIM(parcel_id)     AS parcel_id,
        TRIM(sorter_id)     AS sorter_id,
        TRIM(chute_id)      AS chute_id,
        TRIM(entry_time)    AS entry_time,
        TRIM(result)        AS result,
        current_timestamp() AS _updated_at
    FROM deduped
) AS source
ON target.event_id = source.event_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *

## STATUS_HISTORY

In [0]:
MERGE INTO parcel_sorting_hub.silver.status_history AS target
USING (
    WITH bronze_new_data AS (
        SELECT 
            *
        FROM parcel_sorting_hub.bronze.status_history
        WHERE _ingested_at > (
            SELECT COALESCE(MAX(_updated_at), '2026-04-30')
            FROM parcel_sorting_hub.silver.status_history
        )
            AND history_id IS NOT NULL
            AND TRIM(history_id) != ''
            AND TRIM(history_id) != 'null'
            AND status_name IN ('RECEIVED', 'SORTED', 'LOADED', 'EXCEPTIONS')
    ),

    deduped AS (
        SELECT 
            * EXCEPT (row_num_id)
        FROM (
            SELECT *,
                ROW_NUMBER() OVER (PARTITION BY TRIM(history_id) ORDER BY _ingested_at DESC) AS row_num_id,
                ROW_NUMBER() OVER (PARTITION BY 
                    TRIM(history_id),
                    TRIM(parcel_id),
                    TRIM(status_name),
                    TRIM(change_timestamp)
                ORDER BY _ingested_at DESC) AS row_dup
            FROM bronze_new_data
        )
        WHERE row_num_id = 1 AND row_dup = 1
    )
    
    SELECT
        TRIM(history_id)    AS history_id,
        TRIM(parcel_id)     AS parcel_id,
        TRIM(status_name)        AS status,
        TRIM(change_timestamp) AS change_timestamp,
        current_timestamp() AS _updated_at
    FROM deduped

) AS source
ON target.history_id = source.history_id

WHEN MATCHED THEN 
    UPDATE SET *

WHEN NOT MATCHED THEN 
    INSERT *